# Logistic Regression - Theory, from scratch & scikit-learn comparison

## 1. Introduction

Logistic Regression is a **supervised learning** algorithm used for **binary classification** problems.

Unlike Linear Regression, which predicts a continuous value, Logistic Regression **predicts a probability that an input belongs to class 1**.

It models: 

$P(y = 1 \mid X)$

The output is constrained between 0 and 1 using the sigmoid function.

### 1.1. Scalar form

For a single observation with features $x_1, x_2, \dots, x_n $ the linear model is:

$z = \theta_0 + \theta_1 x_1 + \theta_2 x_2 + \dots + \theta_n x_n$

Logistic Regression applies the sigmoid function:

$\hat{y} = \sigma(z)$

Where the sigmoid function is:

$\sigma(z) = \frac{1}{1 + e^{-z}}$

Therefore:

$\hat{y} = \frac{1}{1 + e^{-(\theta_0 + \theta_1 x_1 + \dots + \theta_n x_n)}} = P(y = 1 \mid X)$

#### Decision Rule

$\hat{y}_{class} =
\begin{cases}
1 & \text{if } \hat{y} \ge 0.5, \
0 & \text{otherwise}
\end{cases}
$

### 1.2. Vector / Matrix form

Let:

* $X \in \mathbb{R}^{m \times n}$
* $\theta \in \mathbb{R}^{n}$

The linear combination is:

$z = X\theta$

Predicted probabilities:

$\hat{y} = \sigma(X\theta)$

Sigmoid is applied element-wise.

## 2. Goal of Logistic Regression

### 2.1 Probabilistic Model

We assume:

$y_i \sim \text{Bernoulli}(p_i)$

Where:

$p_i = \sigma(\theta^T x_i)$

The probability of observing $y_i$ is:

$P(y_i \mid x_i) = p_i^{y_i} (1 - p_i)^{1 - y_i}$

### 2.2 Likelihood Function

For the full dataset:

$L(\theta) = \prod_{i=1}^{m}p_i^{y_i} (1 - p_i)^{1 - y_i}$

We want to **maximize the likelihood**.

Instead, we minimize the **negative log-likelihood**.

### 2.3 Cost Function (Binary Cross-Entropy)

Taking the negative log:

$J(\theta) = - \frac{1}{m}\sum_{i=1}^{m} \left[ y_i \log(p_i) - (1 - y_i) \log(1 - p_i) \right]$

This is called:

* Binary Cross-Entropy
* Log Loss

#### Why Not Mean Squared Error?

Using MSE:

$J(\theta) = \frac{1}{m} \sum (y - \hat{y})^2$

With a sigmoid function leads to a **non-convex cost function**.

Cross-entropy gives a **convex optimization problem**, ensuring a unique global minimum.

### 2.4 Gradient of the Cost Function

The gradient is:

$\nabla J(\theta) = \frac{1}{m}X^T (\hat{y} - y)$

This elegant form makes gradient descent efficient.

### 2.5 Optimization

Common optimization methods:

* Batch Gradient Descent
* Stochastic Gradient Descent (SGD)
* Newton’s Method
* LBFGS (used in scikit-learn)

## 3. Log-Odds Interpretation

Logistic Regression models the **log-odds**:

$\log \left(
\frac{p}{1-p}
\right)
\theta^T x
$

This is called the **logit function**.

**Interpretation**: Each coefficient $\theta_j$ represents the change in **log-odds** when feature $x_j$ increases by one unit.

## 4. Assumptions

1. Binary dependent variable
2. Independent observations
3. Linear relationship between features and log-odds
4. No strong multicollinearity
5. Large sample size preferred

## 5. Regularization

To prevent overfitting, we add a penalty term.

### 5.1. L2 Regularization

$J(\theta) = \text{LogLoss} + \lambda |\theta|^2$

### 5.2. L1 Regularization

$J(\theta) = \text{LogLoss} + \lambda |\theta|_1$

* L1 → sparsity
* L2 → smooth shrinkage

## 6. Implementation from Scratch (NumPy)

## 7. Implementation with Scikit-Learn

In [ ]:
# Librairies imports
import numpy as np
import matplotlib.pyplot as plt

from sklearn.datasets import make_classification
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, classification_report

In [ ]:
# Dataset generation with 2 features
X, y = make_classification(
    n_samples=500,
    n_features=2,
    n_redundant=0,
    n_informative=2,
    n_clusters_per_class=1,
    random_state=42
)

# Train-test split
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

In [ ]:
# Train the model
model = LogisticRegression()
model.fit(X_train, y_train)

In [ ]:
# Prediction and evaluation
y_pred = model.predict(X_test)
y_proba = model.predict_proba(X_test)[:, 1]

print("Accuracy:", accuracy_score(y_test, y_pred))
print("\nClassification Report:\n", classification_report(y_test, y_pred))

print("\nModel Parameters")
print("Weights:", model.coef_)
print("Bias:", model.intercept_)

In [ ]:
# Plotting training data with different colors for each class
plt.figure()

plt.scatter(X_train[y_train == 0][:, 0],
            X_train[y_train == 0][:, 1],
            label="Class 0")

plt.scatter(X_train[y_train == 1][:, 0],
            X_train[y_train == 1][:, 1],
            label="Class 1")

plt.xlabel("Feature 1")
plt.ylabel("Feature 2")
plt.title("Training Data")
plt.legend()

plt.show()


In [ ]:
# Decision boundary visualization
# Create mesh grid
x_min, x_max = X[:, 0].min() - 1, X[:, 0].max() + 1
y_min, y_max = X[:, 1].min() - 1, X[:, 1].max() + 1

xx, yy = np.meshgrid(
    np.linspace(x_min, x_max, 300),
    np.linspace(y_min, y_max, 300)
)

# Flatten grid
grid = np.c_[xx.ravel(), yy.ravel()]

# Predict probabilities
probs = model.predict_proba(grid)[:, 1]
probs = probs.reshape(xx.shape)

# Plot probability surface
plt.figure()

plt.contourf(xx, yy, probs, alpha=0.3)
plt.contour(xx, yy, probs, levels=[0.5])

# Plot training points
plt.scatter(X_train[:, 0], X_train[:, 1], c=y_train)

plt.xlabel("Feature 1")
plt.ylabel("Feature 2")
plt.title("Logistic Regression Decision Boundary")

plt.show()